# PneumoniaMNIST — Diffusion Model

Train and develop a **Diffusion Model** from scratch to generate 28×28 grayscale chest X-rays.
This notebook exports samples to the **Diffusion Model leaderboard**. VAE and diffusion entries have
independent best scores and ranks, and both boards appear together on the student/teacher screen.

Save a copy to Drive and select **Runtime → Change runtime type → T4 GPU** if available.
Your task is to develop the architecture and training recipe, record experiments, and discuss sample quality/diversity.
The starter trains one model; it does not automatically choose architectures or compare model families.

Dataset: [MedMNIST v2, Yang et al. (2023)](https://www.nature.com/articles/s41597-022-01721-8),
derived from [Kermany et al. (2018)](https://doi.org/10.1016/j.cell.2018.02.010).
PneumoniaMNIST uses CC BY 4.0. This exercise uses its standard 28×28 grayscale images.

In [ ]:
%pip install -q medmnist==3.0.2

In [ ]:
import hashlib
import importlib
from pathlib import Path
import random
import urllib.request

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import PneumoniaMNIST
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path("data").mkdir(exist_ok=True)
transform = transforms.ToTensor()  # one grayscale channel, values in [0, 1]
train_data = PneumoniaMNIST(split="train", root="data", download=True, size=28, transform=transform)
validation_data = PneumoniaMNIST(split="val", root="data", download=False, size=28, transform=transform)
BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("Device:", device, "| Training images:", len(train_data), "| Validation:", len(validation_data))

## Generation benchmark

The server ranks a fixed-size batch of **256 new unconditional samples** by **RBF feature MMD²** against
a fixed held-out reference distribution. The frozen PneumoniaMNIST feature encoder, training-feature
normalisation and kernel bandwidth are identical for both boards. Lower scores are better.
This is a domain-feature distribution-matching benchmark, **not Inception-based FID or KID**.

The V-statistic includes diagonal terms, so it is non-negative apart from numerical rounding.
Also inspect feature diversity and image galleries: a score alone does not establish clinical realism.
Train only on training images and use validation losses to guide development. Do not copy images,
submit reconstructions, use pretrained models, or hand-select favourable generated samples.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/jamie-duell/interactive_coding-main/main/examples/data"
SOURCE_URL = "https://raw.githubusercontent.com/jamie-duell/interactive_coding-main/main/backend/app/challenges/generative_models"
for name, url in [
    ("encoder_model.npz", f"{DATA_URL}/generative_encoder_model.npz"),
    ("reference_distribution.npz", f"{DATA_URL}/generative_reference_distribution.npz"),
    ("generation_reference.py", f"{SOURCE_URL}/reference.py"),
]:
    urllib.request.urlretrieve(url, name)
import generation_reference
importlib.reload(generation_reference)
BENCHMARK_SHA256 = generation_reference.benchmark_sha256()
assert BENCHMARK_SHA256 == "1c0fcefb3ee2ac7a4d11168130c7466e40268f670c6cecd4dc963e39b66f637e", "Use the current notebook and benchmark together."

## Your diffusion model — edit and experiment

The starter is a small time-conditioned convolutional noise predictor. It receives a noisy
image and timestep and predicts the noise added at that step. Develop the architecture,
time conditioning, number of steps, beta schedule and training settings.
Inputs are `(N, 1, 28, 28)`; output is the same shape. Rerun setup and training after edits.
[DDPM paper](https://arxiv.org/abs/2006.11239).

In [ ]:
TIMESTEPS = 100
# Scale the linear DDPM schedule to 100 steps so the final images are nearly pure noise.
BETAS = (torch.linspace(1e-4, 0.02, TIMESTEPS, device=device) * (1000 / TIMESTEPS)).clamp(max=0.999)
ALPHAS = 1.0 - BETAS
ALPHA_BARS = torch.cumprod(ALPHAS, dim=0)


class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.time = nn.Sequential(nn.Embedding(TIMESTEPS, 32), nn.Linear(32, 32), nn.SiLU())
        self.input = nn.Conv2d(1, 32, 3, padding=1)
        self.blocks = nn.Sequential(nn.SiLU(), nn.Conv2d(32, 32, 3, padding=1), nn.SiLU(),
                                    nn.Conv2d(32, 32, 3, padding=1), nn.SiLU(), nn.Conv2d(32, 1, 3, padding=1))

    def forward(self, images, timesteps):
        hidden = self.input(images) + self.time(timesteps)[:, :, None, None]
        return self.blocks(hidden)


model = Denoiser().to(device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Train: noise prediction

Each training image is scaled to `[-1, 1]`. Draw a random timestep and Gaussian noise,
form the noisy image using the cumulative alpha schedule, then minimise noise-prediction MSE.
Validation uses a repeated random seed to make checkpoints comparable.
Diffusion training usually needs longer runs than the starter; begin with five epochs to check
the pipeline, then develop your architecture and training budget.

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
best_state, best_validation = None, float("inf")
history = []

def noise_loss(images):
    clean = images * 2 - 1
    t = torch.randint(0, TIMESTEPS, (len(images),), device=device)
    noise = torch.randn_like(clean)
    alpha_bar = ALPHA_BARS[t][:, None, None, None]
    noisy = alpha_bar.sqrt() * clean + (1 - alpha_bar).sqrt() * noise
    return F.mse_loss(model(noisy, t), noise)

for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        optimizer.zero_grad()
        loss = noise_loss(images)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(images)
    model.eval()
    validation = 0.0
    with torch.random.fork_rng(devices=[device.index or 0] if device.type == "cuda" else []):
        torch.manual_seed(123)
        with torch.inference_mode():
            for images, _ in validation_loader:
                validation += noise_loss(images.to(device)).item() * len(images)
    validation /= len(validation_data)
    history.append({"epoch": epoch + 1, "training_loss": total / len(train_data), "validation_loss": validation})
    if validation < best_validation:
        best_validation = validation
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    print(f"Epoch {epoch+1}: train {total/len(train_data):.4f}, validation {validation:.4f}")

assert best_state is not None, "Train for at least one epoch."
model.load_state_dict(best_state)
model.eval()

## Record your experiments

| Architecture / settings | Epochs | Validation loss | Feature MMD² | Feature diversity | What did you learn? |
|---|---:|---:|---:|---:|---|
| Starter | | | | | |
| Your change | | | | | |
| Your final choice | | | | | |

Discuss quality, diversity, training cost and generation speed. If you try both families,
your arena page keeps separate best results on the two leaderboards.

## Generate 256 new samples and evaluate

Use the fixed seed, export the first 256 generated images, and inspect the first 16.
Do not select favourable outputs or copy/reconstruct training, validation or test images.
The server recomputes this same quality score from your uploaded sample batch.

In [ ]:
SAMPLING_SEED = 2026  # fixed sampling seed for the exercise
generator = torch.Generator(device=device).manual_seed(SAMPLING_SEED)
parts = []
with torch.inference_mode():
    for start in range(0, 256, 32):
        x = torch.randn(32, 1, 28, 28, device=device, generator=generator)
        for step in reversed(range(TIMESTEPS)):
            t = torch.full((len(x),), step, dtype=torch.long, device=device)
            predicted_noise = model(x, t)
            alpha, alpha_bar, beta = ALPHAS[step], ALPHA_BARS[step], BETAS[step]
            # Predict x0, clip to the image range, then use the DDPM posterior mean/variance.
            x0 = ((x - (1 - alpha_bar).sqrt() * predicted_noise) / alpha_bar.sqrt()).clamp(-1, 1)
            previous_bar = ALPHA_BARS[step - 1] if step > 0 else torch.ones((), device=device)
            mean = (previous_bar.sqrt() * beta / (1 - alpha_bar)) * x0
            mean += (alpha.sqrt() * (1 - previous_bar) / (1 - alpha_bar)) * x
            if step > 0:
                variance = beta * (1 - previous_bar) / (1 - alpha_bar)
                noise = torch.randn(x.shape, device=device, generator=generator)
                x = mean + variance.sqrt() * noise
            else:
                x = mean
        parts.append(((x + 1) / 2).clamp(0, 1).cpu())
samples = torch.cat(parts)[:, 0].numpy().astype(np.float32)
assert samples.shape == (256, 28, 28) and np.isfinite(samples).all()
quality, diversity = generation_reference.score(samples)
print(f"Feature MMD²: {quality:.4f} (lower is better); feature diversity: {diversity:.3f}")
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for ax, image in zip(axes.flat, samples[:16]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Export your experiment

Keep the model family below correct; it determines which leaderboard receives the submission.
Give your architecture/settings a short description. The server accepts samples only and cannot
verify the architecture from images, so keep the notebook as your experiment record.

In [ ]:
MODEL_FAMILY = "diffusion"
DESCRIPTION = f"Diffusion Model starter; {EPOCHS} epochs"  # update to match your experiment
assert 1 <= len(DESCRIPTION) <= 128
np.savez_compressed(f"{MODEL_FAMILY}_samples.npz", samples=samples.astype(np.float32),
                    model_family=np.array(MODEL_FAMILY), benchmark_sha256=np.array(BENCHMARK_SHA256),
                    description=np.array(DESCRIPTION))
print(f"Upload {MODEL_FAMILY}_samples.npz on your arena student page.")
try:
    from google.colab import files
    files.download(f"{MODEL_FAMILY}_samples.npz")
except ImportError:
    print("Outside Colab: use the notebook file browser.")